**Group information**

| Family name | First name | Email address |
| ----------- | ---------- | ------------- |
|             |            |               |
|             |            |               |
|             |            |               |

# Image - Practice

In this tutorial we explore how computer vision models can be used to predict emotions from facial images. The labelled dataset available on [Kaggle](https://www.kaggle.com/datasets/jonathanoheix/face-expression-recognition-dataset) contains 35,887 images of dimensions $48 \times 48 \times 3$, representing faces expressing 7 fundamental emotions including anger, disgust, fear, happiness, neutral, sadness, and surprise. We aim to approximate the function mapping each image to a set of conditional probabilities corresponding to each emotion using a simple convolutional neural network.

In [1]:
# Packages
import numpy as np
import pandas as pd
import os
import shutil
import torch
import torchinfo

from matplotlib import pyplot as plt
from sklearn import metrics
from torch import nn, optim, utils
from torchvision import io, transforms
from tqdm import tqdm
from urllib import request

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
device = torch.device(device)

# Utilities
emotions = {0:'Anger', 1:'Disgust', 2:'Fear', 3:'Happiness', 4:'Neutral', 5:'Sadness', 6:'Surprise'}

In [2]:
# Utilities
def plot_image(image:torch.Tensor, title:str='', cmap:str='gray', figsize=(5, 5)) -> None:
    image   = torch.einsum('dhw -> hwd', image)
    fig, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=15)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    plt.close()

def download_data():
    if os.getcwd().endswith('/data'):
        print('Data folder already exists')
    else:
        request.urlretrieve('https://www.dropbox.com/scl/fo/8ls3rbwheo8rvactkvdx0/AFVMKWwDTisjrMxDyApPAKY?rlkey=nysbl3yw84jsqk9ry1tz9xwd1&dl=1', 'data.zip')
        shutil.unpack_archive('data.zip', 'data')
        os.remove('data.zip')
        os.chdir('data')

In [3]:
# Downloads dataset
download_data()

**1. Load data:** Use `torch.load` to load the images and labels tensors.

**2. Descriptive statistics:** Check the dimensions of images and labels tensors using the `size` methods, calculate label frequencies, and display a few image-label examples using the provided `plot_image` function.

**3. Data formatting:** Convert the images tensor to `torch.float32` and normalise them to the `[0, 1]` range. Convert the labels tensor to `torch.int64` (or `torch.long`) as required by the cross-entropy loss function.

**4. Train-test split:** Randomly partition images and labels into a training sample (75%) and a test sample (25%). Create PyTorch datasets (`utils.data.TensorDataset`) and data loaders (`utils.data.DataLoader`).

**5. Model structure:** Define a PyTorch model class for a simple convolutional network with the following structure:

- Three blocks of layers with $d = 8, 16, 32$ convolutional filters, respectively:
    - Convolution (`nn.Conv2d`) with $d$ filters of size $3 \times 3$
    - ReLU activation (`nn.ReLU`)
    - $2 \times 2$ max pooling (`nn.MaxPool2d`)
- A global average pooling layer (`nn.AdaptiveAvgPool2d`)
- A linear output layer (`nn.Linear`)

Instantiate the model and print the model structure.

Note: For numerical stability, PyTorch loss functions expect raw logit scores rather than probability distributions. The softmax transformation is applied internally within the loss function.

**6. Loss function and optimiser:** Define the appropriate loss function (`nn.CrossEntropyLoss`) and an optimisation algorithm (e.g. `optim.AdamW`).

**7. Model training:** Write a PyTorch training loop to estimate the model parameters using the training sample, with a maximum of 100 epochs. 

Note: When using `cuda` or `mps` devices, make sure to move both the model and the input tensors to the appropriate device, and to move back the output tensor to `cpu`.

**8. Model evaluation:** Write a PyTorch evaluation loop to assess the model's generalisation performance on the test sample.

**9. Confusion matrix:** Display a sample of test images alongside their predicted labels. Compute the confusion matrix and interpret the results.

**10. Prediction:** Take a selfie and run it through the model to predict the emotion.

**Bonus (advanced)**:  Use a PyTorch hook to extract the output of the last hidden layer. Apply a distance-preserving dimensionality reduction technique (e.g. UMAP or t-SNE) to visualise the resulting embedding space.